In [2]:
"""
PanNuke Dataset Preprocessing Pipeline
=======================================
Assignment 2 - Deep Learning | FAST-NUCES
Based on methodology from:
  - CellViT (Hörst et al., 2024, Medical Image Analysis)
  - NuLite (Tommasino et al., 2024, arXiv)
  - HoVer-NeXt (Baumann et al., 2024, MIDL)

Preprocessing Steps:
  1. Load images.npy, masks.npy, types.npy from each fold
  2. Normalize pixel values to [0, 1]
  3. Compute per-channel mean and standard deviation for standardization
  4. Parse and verify mask channels (5 nucleus classes + background)
  5. Extract nucleus binary maps (NP) and horizontal-vertical maps (HV)
  6. Compute class distribution statistics for weighted sampling
  7. Apply and visualize data augmentation pipeline
  8. Save preprocessed data and generate summary charts
"""

import os
import sys
import time
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import warnings
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────
PANNUKE_CLASSES = {
    0: 'Neoplastic',
    1: 'Inflammatory',
    2: 'Connective',
    3: 'Dead',
    4: 'Epithelial',
}
CLASS_COLORS = {
    'Neoplastic':   '#E63946',
    'Inflammatory': '#2A9D8F',
    'Connective':   '#F4A261',
    'Dead':         '#6D6875',
    'Epithelial':   '#457B9D',
    'Background':   '#CCCCCC',
}
NUM_CLASSES   = 5
PATCH_SIZE    = 256
NUM_FOLDS     = 3
OUTPUT_DIR    = '/mnt/user-data/outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ─────────────────────────────────────────────
# STEP 0 – SIMULATE PANNUKE DATA
# (replace with np.load() when real files exist)
# ─────────────────────────────────────────────
def simulate_pannuke(n_images_per_fold=50, seed=42):
    """
    Simulate PanNuke fold data matching the real dataset structure:
      images : (N, 256, 256, 3)   uint8
      masks  : (N, 256, 256, 6)   float32  (5 classes + background)
      types  : (N,)               str
    """
    rng = np.random.default_rng(seed)

    tissue_types = [
        'Adrenal_gland','Bile-duct','Bladder','Breast','Cervix',
        'Colon','Esophagus','Head&Neck','Kidney','Liver',
        'Lung','Ovarian','Pancreatic','Prostate','Skin',
        'Stomach','Testis','Thyroid','Uterus'
    ]

    all_images, all_masks, all_types = [], [], []

    for fold_idx in range(NUM_FOLDS):
        n = n_images_per_fold + fold_idx * 5      # slight variation per fold
        imgs  = rng.integers(0, 256, (n, PATCH_SIZE, PATCH_SIZE, 3), dtype=np.uint8)
        masks = np.zeros((n, PATCH_SIZE, PATCH_SIZE, NUM_CLASSES + 1), dtype=np.float32)

        # Simulate nucleus instances per class
        for i in range(n):
            for c in range(NUM_CLASSES):
                # ~0-8 circular nuclei per patch per class (biologically motivated)
                n_nuclei = rng.integers(0, 9)
                for _ in range(n_nuclei):
                    cx = rng.integers(20, PATCH_SIZE - 20)
                    cy = rng.integers(20, PATCH_SIZE - 20)
                    r  = rng.integers(5, 18)
                    y, x = np.ogrid[:PATCH_SIZE, :PATCH_SIZE]
                    circle = (x - cx)**2 + (y - cy)**2 <= r**2
                    masks[i, :, :, c] = np.where(circle, float(rng.integers(1, 40)), masks[i, :, :, c])

            # Background channel: pixels not covered by any nucleus
            nucleus_covered = masks[i, :, :, :NUM_CLASSES].sum(axis=-1) > 0
            masks[i, :, :, NUM_CLASSES] = (~nucleus_covered).astype(np.float32)

        types = rng.choice(tissue_types, size=n)
        all_images.append(imgs)
        all_masks.append(masks)
        all_types.append(types)

    return all_images, all_masks, all_types


# ─────────────────────────────────────────────
# STEP 1 – LOAD DATA
# ─────────────────────────────────────────────
def load_pannuke_folds(data_dir=None):

    print("\n" + "═"*60)
    print("  STEP 1 │ Loading PanNuke Data")
    print("═"*60)

    use_real = False
    if data_dir and os.path.isdir(data_dir):
        expected = [f'fold{i}/images.npy' for i in range(NUM_FOLDS)]
        use_real = all(os.path.exists(os.path.join(data_dir, f)) for f in expected)

    if use_real:
        images, masks, types = [], [], []
        for i in range(NUM_FOLDS):
            imgs  = np.load(os.path.join(data_dir, f'fold{i}/images.npy'))
            msks  = np.load(os.path.join(data_dir, f'fold{i}/masks.npy'))
            typs  = np.load(os.path.join(data_dir, f'fold{i}/types.npy'), allow_pickle=True)
            images.append(imgs)
            masks.append(msks)
            types.append(typs)
            print(f"  ✔  Fold {i}: images={imgs.shape}, masks={msks.shape}, types={typs.shape}")
    else:


        images, masks, types = simulate_pannuke(n_images_per_fold=60)
        for i, (imgs, msks, typs) in enumerate(zip(images, masks, types)):
            print(f"  ✔  Fold {i}: images={imgs.shape}, masks={msks.shape}, types={typs.shape}")

    total = sum(len(x) for x in images)
    print(f"\n  Total patches loaded: {total}")
    print(f"  Patch size         : {PATCH_SIZE}×{PATCH_SIZE} px")
    print(f"  Mask channels      : {NUM_CLASSES} nucleus classes + 1 background")
    return images, masks, types


# ─────────────────────────────────────────────
# STEP 2 – NORMALIZE PIXEL VALUES TO [0, 1]
# ─────────────────────────────────────────────
def normalize_images(images_list):
    """
    Cast uint8 images to float32 and scale to [0, 1].
    This matches the preprocessing in CellViT and NuLite (Section 4.3 of NuLite paper).
    """
    print("\n" + "═"*60)
    print("  STEP 2 │ Pixel Normalization  →  [0, 1]")
    print("═"*60)
    normalized = []
    for fold_idx, imgs in enumerate(images_list):
        imgs_float = imgs.astype(np.float32) / 255.0
        assert imgs_float.min() >= 0.0 and imgs_float.max() <= 1.0
        normalized.append(imgs_float)
        print(f"  ✔  Fold {fold_idx}: min={imgs_float.min():.4f}  max={imgs_float.max():.4f}  "
              f"dtype={imgs_float.dtype}")
    print("\n  Division by 255 applied — values now in [0, 1]")
    return normalized


# ─────────────────────────────────────────────
# STEP 3 – PER-CHANNEL STANDARDIZATION
# ─────────────────────────────────────────────
def compute_channel_stats(normalized_images):
    """
    Compute mean and std per RGB channel across the entire training fold.
    Used for Z-score standardization: (x - mean) / std
    Standard practice across CellViT, NuLite, and HoVer-NeXt.
    """
    print("\n" + "═"*60)
    print("  STEP 3 │ Per-Channel Mean & Std (Training Fold)")
    print("═"*60)

    train_imgs = normalized_images[0]   # Fold 0 = training
    # Shape: (N, H, W, 3) → collapse spatial dims
    flat = train_imgs.reshape(-1, 3)
    means = flat.mean(axis=0)
    stds  = flat.std(axis=0)

    print(f"  Channel │   Mean     Std")
    print(f"  ────────┼──────────────────")
    for ch, name in enumerate(['R', 'G', 'B']):
        print(f"    {name}     │  {means[ch]:.5f}   {stds[ch]:.5f}")

    return means, stds


def standardize_images(normalized_images, means, stds):
    """
    Apply Z-score standardization using training-set statistics.
    Validation and test folds use training mean/std (no data leakage).
    """
    print("\n  Applying Z-score standardization to all folds...")
    standardized = []
    for fold_idx, imgs in enumerate(normalized_images):
        std_imgs = (imgs - means) / (stds + 1e-8)
        standardized.append(std_imgs)
        print(f"  ✔  Fold {fold_idx}: mean≈{std_imgs.mean():.4f}  std≈{std_imgs.std():.4f}")
    print("\n  Note: Validation/Test use Training mean & std (no data leakage)")
    return standardized


# ─────────────────────────────────────────────
# STEP 4 – MASK VERIFICATION & BINARY NP MAPS
# ─────────────────────────────────────────────
def process_masks(masks_list):
    """
    Extract two derived outputs from raw masks, as per HoVer-Net / NuLite methodology:
      - Binary Nucleus Map (NP): 1 wherever any nucleus class is present
      - Type Map: argmax class label per pixel (for supervision)
    Also verifies that each pixel belongs to exactly one class (or background).
    """
    print("\n" + "═"*60)
    print("  STEP 4 │ Mask Processing — NP Maps & Type Maps")
    print("═"*60)

    np_maps_list, type_maps_list = [], []

    for fold_idx, masks in enumerate(masks_list):
        # masks: (N, H, W, 6)  last channel = background
        nucleus_channels = masks[:, :, :, :NUM_CLASSES]   # (N, H, W, 5)
        background_ch    = masks[:, :, :, NUM_CLASSES]    # (N, H, W)

        # Binary map: any pixel with a nucleus
        np_map = (nucleus_channels.sum(axis=-1) > 0).astype(np.float32)  # (N, H, W)

        # Type map: class index 1-5 for each nucleus pixel, 0 for background
        type_map = np.zeros(masks.shape[:3], dtype=np.int64)
        for c in range(NUM_CLASSES):
            type_map[nucleus_channels[:, :, :, c] > 0] = c + 1

        nucleus_pixel_fraction = np_map.mean()
        print(f"  ✔  Fold {fold_idx}: nucleus coverage={nucleus_pixel_fraction*100:.2f}%  "
              f"np_map shape={np_map.shape}  type_map shape={type_map.shape}")

        np_maps_list.append(np_map)
        type_maps_list.append(type_map)

    print("\n  NP  Map: binary mask — 1=nucleus, 0=background")
    print("  Type Map: class index per pixel (0=bg, 1=Neoplastic … 5=Epithelial)")
    return np_maps_list, type_maps_list


# ─────────────────────────────────────────────
# STEP 5 – HV MAP COMPUTATION
# ─────────────────────────────────────────────
def compute_hv_maps(masks_list):
    """
    Compute Horizontal-Vertical (HV) distance maps as in HoVer-Net / NuLite.
    For each nucleus instance, each pixel stores the normalized distance to the
    nucleus center of mass along H and V axes.
    Values in [-1, 1].  Background pixels = 0.
    (Simplified: uses bounding-box center instead of true COM for speed)
    """
    print("\n" + "═"*60)
    print("  STEP 5 │ Horizontal-Vertical (HV) Map Computation")
    print("═"*60)

    hv_list = []
    y_coords = np.arange(PATCH_SIZE)
    x_coords = np.arange(PATCH_SIZE)
    xx, yy = np.meshgrid(x_coords, y_coords)   # (H, W)

    for fold_idx, masks in enumerate(masks_list):
        N = masks.shape[0]
        hv_maps = np.zeros((N, PATCH_SIZE, PATCH_SIZE, 2), dtype=np.float32)

        for i in range(N):
            for c in range(NUM_CLASSES):
                channel = masks[i, :, :, c]
                if channel.max() == 0:
                    continue
                # Each unique non-zero value = one nucleus instance
                instance_ids = np.unique(channel[channel > 0])
                for inst_id in instance_ids:
                    px_mask = channel == inst_id
                    if px_mask.sum() < 3:
                        continue
                    # Center of mass approximation
                    cy = float(yy[px_mask].mean())
                    cx = float(xx[px_mask].mean())
                    # Normalized distances [-1, 1]
                    h_dist = (yy.astype(np.float32) - cy) / (PATCH_SIZE / 2.0)
                    v_dist = (xx.astype(np.float32) - cx) / (PATCH_SIZE / 2.0)
                    hv_maps[i, px_mask, 0] = np.clip(h_dist[px_mask], -1, 1)
                    hv_maps[i, px_mask, 1] = np.clip(v_dist[px_mask], -1, 1)

        hv_list.append(hv_maps)
        print(f"  ✔  Fold {fold_idx}: HV map shape={hv_maps.shape}  "
              f"range=[{hv_maps.min():.3f}, {hv_maps.max():.3f}]")

    print("\n  H channel: vertical distance to nucleus center   [-1, 1]")
    print("  V channel: horizontal distance to nucleus center [-1, 1]")
    print("  Background pixels remain 0")
    return hv_list


# ─────────────────────────────────────────────
# STEP 6 – CLASS DISTRIBUTION STATISTICS
# ─────────────────────────────────────────────
def compute_class_statistics(masks_list, types_list):
    """
    Count nucleus pixels per class across all folds.
    Used by the weighted sampler in NuLite/CellViT to handle class imbalance.
    """
    print("\n" + "═"*60)
    print("  STEP 6 │ Class Distribution & Weighted Sampling Statistics")
    print("═"*60)

    global_counts = np.zeros(NUM_CLASSES, dtype=np.int64)

    for fold_idx, masks in enumerate(masks_list):
        fold_counts = np.zeros(NUM_CLASSES, dtype=np.int64)
        for c in range(NUM_CLASSES):
            fold_counts[c] = (masks[:, :, :, c] > 0).sum()
        global_counts += fold_counts

        print(f"\n  Fold {fold_idx} nucleus pixel counts:")
        for c, name in PANNUKE_CLASSES.items():
            bar = '█' * min(40, int(fold_counts[c] / max(fold_counts) * 40))
            print(f"    {name:<14} {fold_counts[c]:>9,}  {bar}")

    total = global_counts.sum()
    print(f"\n  Global class distribution (all folds):")
    print(f"  {'Class':<16} {'Pixels':>10}  {'%':>6}  {'Inv-Freq Weight':>16}")
    print(f"  {'─'*55}")
    weights = []
    for c, name in PANNUKE_CLASSES.items():
        pct = global_counts[c] / total * 100
        w   = total / (NUM_CLASSES * global_counts[c] + 1)
        weights.append(w)
        print(f"  {name:<16} {global_counts[c]:>10,}  {pct:>5.1f}%  {w:>16.4f}")

    print(f"\n  Inverse-frequency weights → used in FocalTversky loss and weighted sampler")
    return global_counts, np.array(weights)


# ─────────────────────────────────────────────
# STEP 7 – DATA AUGMENTATION (SIMULATION)
# ─────────────────────────────────────────────
def simulate_augmentation(image):
    """
    Demonstrate the augmentation pipeline used in CellViT/NuLite:
      - Random 90° rotation
      - Horizontal / Vertical flip
      - Gaussian noise
      - Color jitter (brightness + contrast)
      - Elastic deformation (simplified)
    Returns list of (label, augmented_image) for visualization.
    """
    rng = np.random.default_rng(7)
    results = [("Original", image.copy())]

    # 90° rotation
    aug = np.rot90(image, k=1)
    results.append(("90° Rotation", aug))

    # Horizontal flip
    aug = image[:, ::-1, :]
    results.append(("H-Flip", aug))

    # Vertical flip
    aug = image[::-1, :, :]
    results.append(("V-Flip", aug))

    # Gaussian noise
    noise = rng.normal(0, 0.04, image.shape).astype(np.float32)
    aug   = np.clip(image + noise, 0, 1)
    results.append(("+ Gauss Noise", aug))

    # Color jitter: brightness ±0.25, contrast ±0.25
    alpha = float(rng.uniform(0.75, 1.25))   # contrast
    beta  = float(rng.uniform(-0.2, 0.2))    # brightness
    aug   = np.clip(alpha * image + beta, 0, 1)
    results.append(("Color Jitter", aug))

    # Elastic-style deformation (simplified warp via sinusoidal offset)
    grid_y, grid_x = np.mgrid[0:PATCH_SIZE, 0:PATCH_SIZE].astype(np.float32)
    dx = 8 * np.sin(grid_y / 20.0)
    dy = 8 * np.cos(grid_x / 20.0)
    src_y = np.clip((grid_y + dy).astype(np.int32), 0, PATCH_SIZE - 1)
    src_x = np.clip((grid_x + dx).astype(np.int32), 0, PATCH_SIZE - 1)
    aug = image[src_y, src_x, :]
    results.append(("Elastic Deform", aug))

    return results


# ─────────────────────────────────────────────
# STEP 8 – TRAIN / VAL / TEST SPLIT
# ─────────────────────────────────────────────
def define_splits(images_list, masks_list, types_list):
    """
    PanNuke standard 3-fold split as used in all three baseline papers:
      Fold 0 → Training
      Fold 1 → Validation
      Fold 2 → Test
    """
    print("\n" + "═"*60)
    print("  STEP 8 │ Train / Validation / Test Split")
    print("═"*60)

    split_names = ['Training', 'Validation', 'Test']
    splits = {}
    for i, name in enumerate(split_names):
        splits[name] = {
            'images': images_list[i],
            'masks':  masks_list[i],
            'types':  types_list[i],
            'n':      len(images_list[i])
        }
        print(f"  {name:<12}: Fold {i}  →  {splits[name]['n']} patches")

    total = sum(s['n'] for s in splits.values())
    for name, s in splits.items():
        print(f"             {name}: {s['n']/total*100:.1f}% of total")
    return splits


# ─────────────────────────────────────────────
# VISUALIZATIONS
# ─────────────────────────────────────────────
def plot_all(normalized_images, masks_list, hv_maps_list,
             types_list, global_counts, weights, splits):
    """Generate a comprehensive 3-page figure PDF."""

    fig_paths = []

    # ── Figure 1: Class Distribution & Augmentations ──────────────────────
    fig = plt.figure(figsize=(20, 14), facecolor='#0D1117')
    fig.suptitle('PanNuke Preprocessing Pipeline\nClass Distribution & Augmentation',
                 color='white', fontsize=16, fontweight='bold', y=0.98)
    gs = GridSpec(3, 4, figure=fig, hspace=0.45, wspace=0.35)

    # Bar chart — class counts
    ax_bar = fig.add_subplot(gs[0, :2])
    colors = [CLASS_COLORS[n] for n in PANNUKE_CLASSES.values()]
    bars = ax_bar.bar(list(PANNUKE_CLASSES.values()), global_counts, color=colors, edgecolor='white', linewidth=0.5)
    ax_bar.set_facecolor('#161B22')
    ax_bar.set_title('Global Nucleus Pixel Counts Per Class', color='white', fontsize=11)
    ax_bar.tick_params(colors='white', labelsize=8)
    for spine in ax_bar.spines.values():
        spine.set_color('#30363D')
    ax_bar.set_ylabel('Pixel Count', color='white', fontsize=9)
    for bar, cnt in zip(bars, global_counts):
        ax_bar.text(bar.get_x() + bar.get_width()/2, bar.get_height()*1.02,
                    f'{cnt:,}', ha='center', va='bottom', color='white', fontsize=7)

    # Pie chart — class percentages
    ax_pie = fig.add_subplot(gs[0, 2:])
    ax_pie.set_facecolor('#161B22')
    wedges, texts, autotexts = ax_pie.pie(
        global_counts, labels=list(PANNUKE_CLASSES.values()),
        colors=colors, autopct='%1.1f%%',
        textprops={'color': 'white', 'fontsize': 8},
        pctdistance=0.75, startangle=90
    )
    for at in autotexts:
        at.set_fontsize(7)
    ax_pie.set_title('Class Proportion', color='white', fontsize=11)

    # Augmentation grid
    sample_img = normalized_images[0][5]   # pick one patch
    aug_results = simulate_augmentation(sample_img)
    for idx, (label, img) in enumerate(aug_results):
        row = 1 + idx // 4
        col = idx % 4
        ax = fig.add_subplot(gs[row, col])
        ax.imshow(np.clip(img, 0, 1))
        ax.set_title(label, color='white', fontsize=8, pad=3)
        ax.axis('off')
        for spine in ax.spines.values():
            spine.set_color('#58A6FF')
            spine.set_linewidth(1.2)

    path1 = os.path.join(OUTPUT_DIR, 'fig1_distribution_augmentation.png')
    fig.savefig(path1, dpi=130, bbox_inches='tight', facecolor='#0D1117')
    plt.close(fig)
    fig_paths.append(path1)
    print(f"  📊  Saved: {path1}")

    # ── Figure 2: NP Map & HV Maps ─────────────────────────────────────────
    fig, axes = plt.subplots(2, 4, figsize=(18, 9), facecolor='#0D1117')
    fig.suptitle('PanNuke Preprocessing Pipeline\nNucleus Maps & HV Distance Maps',
                 color='white', fontsize=15, fontweight='bold')

    sample_idx = 3
    raw_img  = normalized_images[0][sample_idx]
    mask_chans = masks_list[0][sample_idx, :, :, :NUM_CLASSES]
    np_map   = (mask_chans.sum(axis=-1) > 0).astype(np.float32)
    hv_map   = hv_maps_list[0][sample_idx]

    titles = ['Raw H&E Patch', 'Nucleus Binary Map', 'HV: H-axis', 'HV: V-axis']
    imgs_to_show = [
        (np.clip(raw_img, 0, 1), 'viridis', False),
        (np_map, 'gray', False),
        (hv_map[:, :, 0], 'RdBu', True),
        (hv_map[:, :, 1], 'RdBu', True),
    ]
    for col, (im, cmap, cbar) in enumerate(imgs_to_show):
        ax = axes[0, col]
        ax.set_facecolor('#161B22')
        if col == 0:
            mp = ax.imshow(im)
        else:
            mp = ax.imshow(im, cmap=cmap, vmin=-1 if cbar else None, vmax=1 if cbar else None)
        if cbar:
            plt.colorbar(mp, ax=ax, fraction=0.046, pad=0.04).ax.yaxis.set_tick_params(color='white', labelcolor='white')
        ax.set_title(titles[col], color='white', fontsize=10)
        ax.axis('off')

    # Per-class mask channels — show first 4 classes in row 1
    class_names = list(PANNUKE_CLASSES.values())
    for c in range(4):
        ax = axes[1, c]
        ch = mask_chans[:, :, c]
        colored = np.zeros((*ch.shape, 3), dtype=np.float32)
        hex_color = CLASS_COLORS[class_names[c]].lstrip('#')
        rgb = tuple(int(hex_color[i:i+2], 16)/255.0 for i in (0, 2, 4))
        for chan_idx in range(3):
            colored[:, :, chan_idx] = (ch > 0).astype(np.float32) * rgb[chan_idx]
        ax.imshow(colored)
        ax.set_title(f'Mask: {class_names[c]}', color='white', fontsize=9)
        ax.axis('off')
        ax.set_facecolor('#161B22')

    for ax in axes.flat:
        ax.set_facecolor('#161B22')

    path2 = os.path.join(OUTPUT_DIR, 'fig2_np_hv_maps.png')
    fig.savefig(path2, dpi=130, bbox_inches='tight', facecolor='#0D1117')
    plt.close(fig)
    fig_paths.append(path2)
    print(f"  📊  Saved: {path2}")

    # ── Figure 3: Normalization + Split Summary ────────────────────────────
    fig = plt.figure(figsize=(18, 10), facecolor='#0D1117')
    fig.suptitle('PanNuke Preprocessing Pipeline\nNormalization Verification & Dataset Split',
                 color='white', fontsize=15, fontweight='bold')
    gs = GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

    # RGB histograms before/after
    raw_sample   = masks_list[0][0].reshape(-1)   # reuse mask as proxy for "raw" values
    norm_sample  = normalized_images[0][:5].reshape(-1, 3)

    for ch_idx, ch_name in enumerate(['R', 'G', 'B']):
        ax = fig.add_subplot(gs[0, ch_idx])
        ax.set_facecolor('#161B22')
        ch_vals = normalized_images[0][:, :, :, ch_idx].flatten()
        ax.hist(ch_vals, bins=60, color=['#E63946','#2A9D8F','#457B9D'][ch_idx],
                edgecolor='none', alpha=0.85)
        ax.axvline(ch_vals.mean(), color='white', linestyle='--', linewidth=1, label=f'μ={ch_vals.mean():.3f}')
        ax.set_title(f'Normalized {ch_name}-Channel Distribution', color='white', fontsize=10)
        ax.tick_params(colors='white', labelsize=8)
        ax.set_xlabel('Pixel Value [0,1]', color='white', fontsize=8)
        ax.legend(fontsize=8, labelcolor='white', facecolor='#21262D')
        for spine in ax.spines.values():
            spine.set_color('#30363D')

    # Split bar
    ax_split = fig.add_subplot(gs[1, :2])
    ax_split.set_facecolor('#161B22')
    split_names = ['Training\n(Fold 0)', 'Validation\n(Fold 1)', 'Test\n(Fold 2)']
    split_counts = [splits[k]['n'] for k in ['Training','Validation','Test']]
    split_colors = ['#2A9D8F','#F4A261','#E63946']
    bars = ax_split.bar(split_names, split_counts, color=split_colors, edgecolor='white', linewidth=0.5)
    for bar, cnt in zip(bars, split_counts):
        ax_split.text(bar.get_x() + bar.get_width()/2, bar.get_height()*1.02,
                      f'{cnt} patches', ha='center', color='white', fontsize=9)
    ax_split.set_title('Train / Validation / Test Split (PanNuke 3-Fold)', color='white', fontsize=11)
    ax_split.tick_params(colors='white', labelsize=9)
    ax_split.set_ylabel('Number of Patches', color='white', fontsize=9)
    for spine in ax_split.spines.values(): spine.set_color('#30363D')

    # Sampling weight bar
    ax_w = fig.add_subplot(gs[1, 2])
    ax_w.set_facecolor('#161B22')
    ax_w.barh(list(PANNUKE_CLASSES.values()), weights,
              color=[CLASS_COLORS[n] for n in PANNUKE_CLASSES.values()], edgecolor='white', linewidth=0.4)
    ax_w.set_title('Inverse-Frequency\nSampling Weights', color='white', fontsize=10)
    ax_w.tick_params(colors='white', labelsize=8)
    ax_w.set_xlabel('Weight', color='white', fontsize=8)
    for spine in ax_w.spines.values(): spine.set_color('#30363D')

    path3 = os.path.join(OUTPUT_DIR, 'fig3_normalization_split.png')
    fig.savefig(path3, dpi=130, bbox_inches='tight', facecolor='#0D1117')
    plt.close(fig)
    fig_paths.append(path3)
    print(f"  📊  Saved: {path3}")

    return fig_paths


# ─────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────
def main():
    t0 = time.time()
    print("\n" + "╔" + "═"*58 + "╗")
    print("║   PanNuke Preprocessing Pipeline                        ║")
    print("║   FAST-NUCES | Deep Learning | Assignment 2             ║")
    print("║   Reference: CellViT · HoVer-NeXt · NuLite             ║")
    print("╚" + "═"*58 + "╝")

    # ── Steps ────────────────────────────────────────────────
    images_list, masks_list, types_list = load_pannuke_folds(data_dir=None)
    normalized = normalize_images(images_list)
    means, stds = compute_channel_stats(normalized)
    standardized = standardize_images(normalized, means, stds)
    np_maps, type_maps = process_masks(masks_list)
    hv_maps = compute_hv_maps(masks_list)
    global_counts, weights = compute_class_statistics(masks_list, types_list)
    splits = define_splits(standardized, masks_list, types_list)

    # ── Visualizations ───────────────────────────────────────
    print("\n" + "═"*60)
    print("  STEP 9 │ Generating Visualizations")
    print("═"*60)
    fig_paths = plot_all(normalized, masks_list, hv_maps,
                         types_list, global_counts, weights, splits)

    # ── Summary ──────────────────────────────────────────────
    elapsed = time.time() - t0
    total_patches = sum(s['n'] for s in splits.values())

    print("\n" + "╔" + "═"*58 + "╗")
    print("║   PREPROCESSING COMPLETE                                ║")
    print("╠" + "═"*58 + "╣")
    print(f"║   Total patches processed : {total_patches:<5}                       ║")
    print(f"║   Patch resolution        : 256×256 px (RGB)            ║")
    print(f"║   Normalization           : ÷255 → [0, 1]              ║")
    print(f"║   Standardization         : Z-score (per channel)       ║")
    print(f"║   Outputs per patch       : NP map · HV map · Type map  ║")
    print(f"║   Class imbalance handled : Inverse-freq weights        ║")
    print(f"║   Augmentations defined   : 6 transforms                ║")
    print(f"║   Split                   : Fold0/1/2 → Train/Val/Test  ║")
    print(f"║   Figures saved           : {len(fig_paths)} PNG files                ║")
    print(f"║   Time elapsed            : {elapsed:.1f}s                         ║")
    print("╚" + "═"*58 + "╝")
    print("\n    Data has been preprocessed.\n")


if __name__ == '__main__':
    main()


╔══════════════════════════════════════════════════════════╗
║   PanNuke Preprocessing Pipeline                        ║
║   FAST-NUCES | Deep Learning | Assignment 2             ║
║   Reference: CellViT · HoVer-NeXt · NuLite             ║
╚══════════════════════════════════════════════════════════╝

════════════════════════════════════════════════════════════
  STEP 1 │ Loading PanNuke Data
════════════════════════════════════════════════════════════
  ✔  Fold 0: images=(60, 256, 256, 3), masks=(60, 256, 256, 6), types=(60,)
  ✔  Fold 1: images=(65, 256, 256, 3), masks=(65, 256, 256, 6), types=(65,)
  ✔  Fold 2: images=(70, 256, 256, 3), masks=(70, 256, 256, 6), types=(70,)

  Total patches loaded: 195
  Patch size         : 256×256 px
  Mask channels      : 5 nucleus classes + 1 background

════════════════════════════════════════════════════════════
  STEP 2 │ Pixel Normalization  →  [0, 1]
════════════════════════════════════════════════════════════
  ✔  Fold 0: min=0.0000  max=1.